In [1]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import torch.nn.functional as F
import tensorflow as tf
import numpy as np
import torch.nn as nn

2026-02-21 13:06:15.289443: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use only GPU 1

In [3]:
%cd /glade/derecho/scratch/lizhili/BiDiffSR_code

/glade/derecho/scratch/lizhili/BiDiffSR_code


/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
from basicsr.utils.options import ordered_yaml
import yaml
file_load = 'options/train/train_BI_DiffSR_x4.yml'
with open(file_load) as f:
    opt = yaml.load(f, Loader=ordered_yaml()[0])

/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/torchvision/transforms/functional_tensor.py:5: UserWarning: The torchvision.transforms.functional_tensor module is deprecated in 0.15 and will be **removed in 0.17**. Please don't rely on it. You probably just need to use APIs in torchvision.transforms.functional or in torchvision.transforms.v2.functional.
  warnings.warn(


In [5]:
opt['is_train'] = True
opt['scale']=16
opt['dist']=False

In [ ]:
from diffglv.models.BI_DiffSR_model import BIDiffSRModel
model = BIDiffSRModel(opt)

In [7]:
def input_pipeline(filename, batch_size, is_shuffle=True, is_train=True, is_repeat=True):
    feature_description = {
        'lres': tf.io.FixedLenFeature([60*60*12], dtype=tf.int64),
        'hres': tf.io.FixedLenFeature([1000*1000*4], dtype=tf.int64),
    }

    def _parse_function(example_proto):
        feature_dict = tf.io.parse_single_example(example_proto, feature_description)
        lres_img = tf.reshape(feature_dict['lres'], [12, 60, 60])
        hres_img = tf.reshape(feature_dict['hres'], [4, 1000, 1000])

        return lres_img, hres_img

    def _augment_function(lres_img, hres_img):
    # Transpose to [H, W, C]
        lres_img = tf.transpose(lres_img, [1, 2, 0])   # [60, 60, 12]
        hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]

        # Randomly choose 0, 90, 180, or 270 degrees
        k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)

        # Apply rotation
        lres_img = tf.image.rot90(lres_img, k=k)
        hres_img = tf.image.rot90(hres_img, k=k)

        # Transpose back to [C, H, W]
        lres_img = tf.transpose(lres_img, [2, 0, 1])   # [12, 60, 60]
        hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]

        return lres_img, hres_img

    dataset = tf.data.TFRecordDataset(filename, buffer_size=8 * 1024 * 1024)
    if is_repeat:
        dataset = dataset.repeat()
    dataset = dataset.map(_parse_function)
    if is_train:
        dataset = dataset.map(_augment_function)
    if is_shuffle:
        dataset = dataset.shuffle(buffer_size=100)
    batch = dataset.batch(batch_size=batch_size)
    return batch

In [ ]:
def load_matched_weights(model, checkpoint_path):
    # Load checkpoint (state_dict)
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint
    # state_dict = state_dict['params']

    # Filter only matching keys
    model_dict = model.state_dict()
    matched_dict = {k: v for k, v in state_dict.items() if k in model_dict and v.shape == model_dict[k].shape}

    # Load the matched weights
    model_dict.update(matched_dict)
    model.load_state_dict(model_dict)

    print(f"✅ Loaded {len(matched_dict)} matching parameters out of {len(model_dict)} total.")

    return model

# model.net_g = load_matched_weights(model.net_g, '/glade/derecho/scratch/lizhili/s2naip/BiDiff_x16_epoch_weights_finetune.pth')
model.net_g = load_matched_weights(model.net_g, '/glade/derecho/scratch/lizhili/s2naip/SR_pretrained_models/BiDiff_x4_epoch_weights_finetune.pth')


In [9]:
def random_crop_pair(lr, hr, crop_size):
    """
    lr, hr: tensors of shape [B, C, H, W]
    crop_size: int
    """
    _, _, H, W = lr.shape
    assert H >= crop_size and W >= crop_size

    top = torch.randint(0, H - crop_size + 1, (1,)).item()
    left = torch.randint(0, W - crop_size + 1, (1,)).item()

    lr_crop = lr[:, :, top:top+crop_size, left:left+crop_size]
    hr_crop = hr[:, :, top:top+crop_size, left:left+crop_size]

    return lr_crop, hr_crop

In [ ]:
import torch.nn.functional as F
import tensorflow as tf
import torch
    
filenames = ['/glade/derecho/scratch/lizhili/s2naip/sr_ds/S2_NAIP_SR_cloudless.tfrecords']
ds = input_pipeline(filenames, batch_size=2, is_shuffle=True, is_train=True, is_repeat=False)
for epoch in range(30):
    print(f'Epoch {epoch}')

    for step, (lr, hr) in enumerate(ds):
        lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')/3500.0
        hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')/255.0

        lr = (lr-0.5)/0.5
        hr = (hr-0.5)/0.5

        lr = F.interpolate(lr, size=(1024, 1024), mode='bilinear', align_corners=False)
        hr = F.interpolate(hr, size=(1024, 1024), mode='bilinear', align_corners=False)
        # lr = F.interpolate(lr, size=(1024, 1024), mode='nearest')
        # hr = F.interpolate(hr, size=(1024, 1024), mode='nearest')
        lr, hr = random_crop_pair(lr, hr, crop_size=512)

        current_iter = step+1
        model.update_learning_rate(current_iter, warmup_iter=opt['train'].get('warmup_iter', -1))
        model.feed_data({'lq': lr, 'gt': hr})
        model.optimize_parameters(current_iter)

        if step % 500 == 0:
            print(f'Step {step}, Loss: {model.get_current_log()}')

    torch.save(model.net_g.state_dict(), '/glade/derecho/scratch/lizhili/s2naip/BiDiff_x16_epoch_weights_finetune.pth') 

In [ ]:

def downstream_run(lres_size ,
                    hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    finetune_tfrecords,
                    finetuned_model,
                    output_tfrecords,
                    model):
    model.net_g = load_matched_weights(model.net_g, '/glade/derecho/scratch/lizhili/s2naip/BiDiff_x16_epoch_weights_finetune.pth')
    # model.net_g = load_matched_weights(model.net_g, finetuned_model)

    num_test = num_sample-num_training

    def input_pipeline_downstream_sr(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):
        feature_description = {
            'lres': tf.io.FixedLenFeature([12*lres_size*lres_size], dtype=tf.int64),
            'hres': tf.io.FixedLenFeature([4*hres_size*hres_size], dtype=tf.int64),
            'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
        }

        @tf.function
        def _parse_function(example_proto):
            feature_dict = tf.io.parse_single_example(example_proto, feature_description)

            lres = feature_dict['lres']
            lres = tf.reshape(lres, [12, lres_size, lres_size])
            lres = tf.cast(lres, tf.float32)

            hres = feature_dict['hres']
            hres = tf.reshape(hres, [4, hres_size, hres_size])
            hres = tf.cast(hres, tf.float32)

            label = feature_dict['label']
            label = tf.reshape(label, [label_size, label_size, 1])
            return lres, hres, label

        @tf.function
        def _augment_function(lres_img, hres_img, label):
        # Transpose to [H, W, C]
            lres_img = tf.transpose(lres_img, [1, 2, 0])   # [60, 60, 12]
            hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]
    
            # Randomly choose 0, 90, 180, or 270 degrees
            k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    
            # Apply rotation
            lres_img = tf.image.rot90(lres_img, k=k)
            hres_img = tf.image.rot90(hres_img, k=k)
            label = tf.image.rot90(label, k=k)
    
            # Transpose back to [C, H, W]
            lres_img = tf.transpose(lres_img, [2, 0, 1])   # [12, 60, 60]
            hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]
    
            return lres_img, hres_img, label

        dataset = tf.data.TFRecordDataset(filename)
        dataset = dataset.skip(skip)
        if take:
            dataset = dataset.take(take)
        if is_repeat:
            dataset = dataset.repeat()
        dataset = dataset.map(_parse_function)
        if is_train:
            dataset = dataset.map(_augment_function)
        if is_shuffle:
            dataset = dataset.shuffle(buffer_size=100)
        batch = dataset.batch(batch_size=batch_size)

        return batch
    # --------------------------------------------
    print('Begin Finetune')
    ds = input_pipeline_downstream_sr(finetune_tfrecords, 2, 0, num_training, is_shuffle=True, is_train=True, is_repeat=False)

    for epoch in range(7):
        print(f'Epoch {epoch}')

        for step, (lr, hr, _) in enumerate(ds):
            lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')/3500.0
            hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')/255.0

            lr = (lr-0.5)/0.5
            hr = (hr-0.5)/0.5

            lr = F.interpolate(lr, size=(1024, 1024), mode='bilinear', align_corners=False)
            hr = F.interpolate(hr, size=(1024, 1024), mode='bilinear', align_corners=False)
            lr, hr = random_crop_pair(lr, hr, crop_size=512)

            current_iter = step+1
            model.update_learning_rate(current_iter, warmup_iter=opt['train'].get('warmup_iter', -1))
            model.feed_data({'lq': lr, 'gt': hr})
            model.optimize_parameters(current_iter)

            if step % 50 == 0:
                print(f'Step {step}, Loss: {model.get_current_log()}')

        torch.save(model.net_g.state_dict(), finetuned_model)

    # --------------------------------------------
    print('Begin Write SR dataset')
    # TFRecord writer setup
    writer = tf.io.TFRecordWriter(output_tfrecords)

    # Serialization function
    def serialize_example(hres, label):
        feature = {
            'hres': tf.train.Feature(int64_list=tf.train.Int64List(value=hres.reshape(-1))),
            'label': tf.train.Feature(int64_list=tf.train.Int64List(value=label.reshape(-1)))
        }
        example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
        return example_proto.SerializeToString()

    # Inference loop
    ds = input_pipeline_downstream_sr(finetune_tfrecords, 4, 1959, num_sample, is_shuffle=False, is_train=False, is_repeat=False)
    for step, (lr, hr, label) in enumerate(ds):
        lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')/3500.0
        hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')/255.0

        lr = F.interpolate(lr, size=(1024, 1024), mode='bilinear', align_corners=False)
        hr = F.interpolate(hr, size=(1024, 1024), mode='bilinear', align_corners=False)

        lr = (lr-0.5)/0.5
        hr = (hr-0.5)/0.5

        model.feed_data({'lq': lr, 'gt': hr})
        model.test()
        output = model.get_current_visuals()['result']
        output = output.detach().cpu().numpy()
        output = np.clip(output*0.5+0.5, 0, 1)
        output = (output * 255).astype(np.uint8)
        label = label.numpy()

        if step < 4:
            lr = lr.detach().cpu().numpy()
        print('batch: ', step)

        for i in range(output.shape[0]):
            print(output[i].shape)
            print(label[i].shape)
            example = serialize_example(output[i], label[i])
            writer.write(example)

            if step < 4:
                fig, axes = plt.subplots(1, 3, figsize=(12, 4))
                lr_show = np.transpose(lr[i], (1, 2, 0))
                axes[0].imshow(lr_show[:, :, 3:0:-1]*0.5+0.5)
                axes[0].set_title('Low-Resolution')
                axes[0].axis('off')

                output_show = np.transpose(output[i], (1, 2, 0))
                axes[1].imshow(output_show[:, :, :3])
                axes[1].set_title('High-Resolution')
                axes[1].axis('off')

                axes[2].imshow(label[i])
                axes[2].set_title('Output')
                axes[2].axis('off')

                plt.show()

    # Close writer
    writer.close()


In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256
label_size = 1000
num_sample = 2000
num_training = 1600
finetune_tfrecords = '/glade/derecho/scratch/lizhili/s2naip/down_ds/USBuildingFootprints.tfrecords'
finetuned_model = '/glade/derecho/scratch/lizhili/s2naip/BiDiff_x16_USBuildingFootprints_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/s2naip/USBuildingFootprints_BiDiff_x16.tfrecords'

downstream_run(lres_size ,
                    hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    finetune_tfrecords,
                    finetuned_model,
                    output_tfrecords,
                    model)

In [ ]:
lres_size = 51
hres_size = 853
hres_size_4x = 256
label_size = 512
num_sample = 2000
num_training = 1600
finetune_tfrecords = '/glade/derecho/scratch/lizhili/s2naip/down_ds/ChesapeakeRSC.tfrecords'
finetuned_model = '/glade/derecho/scratch/lizhili/s2naip/BiDiff_x16_ChesapeakeRSC_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/s2naip/ChesapeakeRSC_BiDiff_x16.tfrecords'


downstream_run(lres_size ,
                    hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    finetune_tfrecords,
                    finetuned_model,
                    output_tfrecords,
                    model)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256
label_size = 1200
num_sample = 2000
num_training = 1600
finetune_tfrecords = '/glade/derecho/scratch/lizhili/s2naip/down_ds/VermontLC.tfrecords'
finetuned_model = '/glade/derecho/scratch/lizhili/s2naip/BiDiff_x16_VermontLC_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/s2naip/VermontLC_BiDiff_x16.tfrecords'

downstream_run(lres_size ,
                    hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    finetune_tfrecords,
                    finetuned_model,
                    output_tfrecords,
                    model)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256
label_size = 1000
num_sample = 2000
num_training = 1600
finetune_tfrecords = '/glade/derecho/scratch/lizhili/s2naip/down_ds/RoadDetections.tfrecords'
finetuned_model = '/glade/derecho/scratch/lizhili/s2naip/BiDiff_x16_RoadDetections_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/s2naip/RoadDetections_BiDiff_x16.tfrecords'

downstream_run(lres_size ,
                    hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    finetune_tfrecords,
                    finetuned_model,
                    output_tfrecords,
                    model)

In [ ]:
lres_size = 26
hres_size = 427
hres_size_4x = 256  # 4x sr image size
label_size = 256 #Vermontlc
num_sample = 2000
num_training = 1600
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/s2naip/down_ds/CHM.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/s2naip/BiDiff_x16_CHM_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/s2naip/CHM_BiDiff_x16.tfrecords'

downstream_run(lres_size ,
                    hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    finetune_tfrecords,
                    finetuned_model,
                    output_tfrecords,
                    model)